# Repository Deep Dive

Questo notebook e la versione tecnica e molto piu dettagliata del walkthrough.

Obiettivi:
- ricostruire il flusso reale di esecuzione
- spiegare i file principali uno per uno
- distinguere fra architettura prevista e snapshot realmente presente
- mostrare dove sono i punti forti e dove sono i punti fragili


## 1. Traccia completa di esecuzione

Il modo piu chiaro per capire la repo e seguirla come se stessimo lanciando un esperimento:

```text
python src/main.py ...
  -> load_config()
  -> configure_logging()
  -> PDDLPlanner(args, config)
  -> planner.setup()
       -> FileManager.find_pddl_files()
       -> ModelManager.load()
       -> PDDLProcessor(...)
  -> planner.run()
       -> PDDLProcessor.process_domain_with_validation()
            -> _process_single_problem()
                 -> _build_prompt()
                 -> ModelManager.iterative_planning_with_validation()
                      -> generate_response()
                      -> formatter()
                      -> validate_plan_from_text()
                 -> save_file(...)
  -> risultati in src/results/
  -> analisi con Results Analysis.ipynb
```

Questa traccia mostra i tre livelli del progetto:
- input PDDL in `src/data/`
- codice orchestratore in `src/core/`, `src/utils/`, `src/prompts/`
- output sperimentali in `src/results/` e notebook di analisi


## 2. `src/main.py`: il punto di ingresso

Questo file non fa planning direttamente: collega configurazione, CLI e orchestratore.

Responsabilita:
- caricare `config.yml`
- definire tutti gli argomenti CLI
- impostare logging e verbosita
- controllare che esistano `problems_path` e `weights_path`
- creare `PDDLPlanner`, chiamare `setup()` e poi `run()`

Argomenti cruciali:
- `--problems_path`, `--weights_path`, `--output_dir`
- `--batch`, `--domain`, `--max_iterations`
- `--sampling`, `--temperature`, `--top_k`, `--max_tokens`
- `--add_system_prompt`, `--cot`, `--include_prompt`, `--skip_special_tokens`
- `--model`

Dettaglio pratico importante:
- `_resolve_output_dir()` aggiunge automaticamente l'alias del modello al path di output, quindi i risultati finiscono in cartelle separate per modello


## 3. `file_manager.py` e `pddl_planner.py`

### `src/core/file_manager.py`
- definisce `DomainBundle`, una dataclass che contiene nome dominio, path del dominio, testo del dominio e lista dei problemi
- `read_file()` legge file testuali con gestione errori
- `save_file()` salva output e crea cartelle se servono
- `find_pddl_files()` e il vero punto chiave: scopre i domini e costruisce i bundle
- `_locate_domain_file()` riconosce sia `domain.pddl` sia `<dominio>_domain.pddl`
- `_collect_problem_files()` tratta tutti gli altri `.pddl` della cartella come istanze

Assunzione implicita:
- una cartella dominio deve contenere un solo file di dominio identificabile; il resto dei `.pddl` e considerato problema

### `src/core/pddl_planner.py`
- e il coordinatore globale
- `setup()` crea `FileManager`, scopre i domini, filtra eventualmente un dominio, carica il modello e crea `PDDLProcessor`
- `run()` sceglie batch o sequenziale e poi aggrega le statistiche
- `_build_processing_kwargs()` centralizza i parametri operativi da passare al processor
- `_log_summary()` scrive nei log il riepilogo finale

In pratica:
- `FileManager` trova e salva file
- `PDDLPlanner` mette insieme tutti gli altri pezzi


## 4. `model_manager.py`: il cuore del loop con LLM

Questo e il modulo piu delicato, perche collega codice Python, pesi del modello, GPU e validazione esterna.

Blocchi principali:
- `__init__()`: salva il path del modello e prova a dedurne il tipo (`phi4`, `llama3`, `gemma3`, `kimi`)
- `load()`: decide CPU/GPU, logga l'inventario GPU, carica modello e tokenizer
- `_load_model()` e `_load_tokenizer()`: wrapper intorno a Hugging Face
- `_ensure_pad_token()`: evita problemi con tokenizer privi di `pad_token`
- `generate_response()`: costruisce input, chiama `model.generate()`, decodifica output
- `_format_messages()`: usa il chat template del tokenizer o un fallback testuale
- `_build_generation_config()`: separa greedy e sampling

Punto distintivo del progetto:
- `iterative_planning_with_validation()` non si ferma a una risposta singola
- genera una proposta di piano
- la passa a `formatter()`
- valida con VAL
- se il piano e invalido, costruisce feedback e riparte fino a `max_iterations`

Dettaglio notevole:
- nel loop iterativo il codice forza `include_prompt=False` per evitare che il prompt torni nella risposta e faccia esplodere il contesto

Fragilita tipiche:
- OOM GPU
- output troppo rumorosi o con reasoning eccessivo
- differenze di formato fra modelli diversi


## 5. `pddl_processor.py`: da dominio e problema a file di risultato

Questo modulo governa il livello per-dominio e per-problema.

Funzioni principali:
- `process_domain_with_validation()`: itera su tutte le istanze di un dominio e costruisce statistiche aggregate
- `batch_process_domains()`: applica la stessa logica a piu domini
- `_process_single_problem()`: legge il problema, costruisce il prompt, chiama il modello e salva l'output
- `_build_prompt()`: compone istruzioni, esempi few-shot, vincoli opzionali e CoT
- `_create_domain_prompt()`: sceglie il prompt giusto per `tetris`, `citycar` o fallback generico
- `_chain_of_thought()` e `_get_validation_feedback_fn()`: scelgono testo specializzato in base al dominio

Dettagli concreti:
- il dominio viene riconosciuto dal nome della cartella, quindi basta che contenga `tetris` o `citycar`
- i risultati vengono salvati con un blocco finale `--- Processing Metadata ---`

Questo design e importante perche rende semplice l'analisi a posteriori senza leggere tutti i log runtime.


## 6. `utils`: configurazione, post-processing e validazione

### `configuration.py`
- cerca `config.yml` nella root del progetto e, in fallback, sotto `src/`
- aggiunge `src` al `sys.path` per rendere piu semplici gli import quando i file vengono eseguiti come script

### `logging_utils.py`
- centralizza il setup del logging
- supporta stdout e log file opzionale

### `common_utils.py`
- contiene helper generici per YAML e directory

### `answer_postprocessor.py`
- e il punto in cui il testo del modello viene trasformato in piano PDDL interpretabile
- `formatter()` produce piano, reasoning, confidence e possibili problemi di formato
- `extract_plan_actions()` usa piu strategie: sezioni `plan`, azioni tra parentesi, liste numerate, righe che iniziano con `(`
- `clean_response_text()` rimuove `<think>`, markdown e scorie varie
- `detect_format_issues()` segnala parentesi sbilanciate, risposta troppo lunga, ecc.

Osservazione importante:
- l'estrazione e euristica, quindi robusta ma non perfetta; puo includere anche frammenti non desiderati se l'output del modello e molto sporco

### `validator.py`
- costruisce il path del binario VAL a partire dalla config
- se non trova il binario nel path previsto, prova a usare il system PATH
- `validate_plan()` lancia VAL via `subprocess`
- `validate_plan_from_text()` crea un file temporaneo `.plan` e poi delega alla funzione precedente

Questa parte e il filtro esterno che decide se il benchmark si fida o no del piano generato dal modello.


## 7. `prompts.py`: il layer di prompt engineering

Questo file concentra tutta la strategia di prompting del progetto.

Contenuti:
- `system_prompt_pddl`: ruolo base del modello come planner PDDL
- helper condivisi: `chain_of_thought_prompt()`, `add_examples_to_prompt()`, `add_constraints_to_prompt()`
- prompt specializzati per `tetris`
- prompt specializzati per `citycar`
- feedback domain-specific dopo errore di validazione
- fallback generici come `generic_pddl_prompt()` e `validation_feedback_prompt()`

Perche e importante:
- qui il benchmark decide quanto aiutare il modello
- gli esempi few-shot forniscono formato e micro-soluzioni
- il feedback non e solo un errore generico, ma una correzione informata dai vincoli del dominio

Quindi la valutazione non misura solo il modello, ma il modello dentro un preciso setup di prompt engineering.


## 8. `src/data/`: capire i due domini

Struttura osservata:
- `citycar/`: 1 file di dominio + 20 istanze
- `tetris/`: 1 file di dominio + 20 istanze

### `citycar`
- dominio di planning urbano
- abbiamo junction, garage, car e road
- si costruiscono o distruggono strade, si avviano auto e si fanno arrivare ai goal finali
- esiste `total-cost`, quindi l'efficienza conta

### `tetris`
- dominio di planning spaziale su griglia
- ci sono pezzi singoli, lineari e a L
- le azioni muovono o riconfigurano i pezzi
- anche qui esiste un costo totale

Questa parte spiega perche la repo usa prompt e feedback separati per i due domini: non sono varianti superficiali dello stesso problema, ma task concettualmente diversi.


## 9. `src/results/` e `Results Analysis.ipynb`

Gerarchia dei risultati:

```text
src/results/<model>/<domain>/instance-xx_plan.txt
```

Ogni file contiene:
- un blocco di testo che il sistema salva come piano
- un blocco finale di metadati con dominio, problema, iterazioni, validita e CoT

Cosa fa `Results Analysis.ipynb`:
- legge tutti i `_plan.txt`
- separa piano e metadati con il marker `--- Processing Metadata ---`
- costruisce una tabella con modello, dominio, problema, validita, lunghezza piano e iterazioni
- produce grafici su successo, lunghezza piani, iterazioni e problemi piu facili o difficili

Nota pratica importante:
- il notebook di analisi ha un `PROJECT_ROOT` hardcoded Linux
- nella sua esecuzione salvata compare un errore per `pandas` mancante

Osservazione fine sulla snapshot:
- in `src/results/gemma3/citycar/instance-02_plan.txt` compaiono anche frammenti che ricordano il problema PDDL, non solo azioni finali pulite
- questo suggerisce che il post-processing e robusto ma non perfettamente conservativo


## 10. `scripts/`, `tests/`, `requirements` e `.gitignore`

### `scripts/`
- sono script Bash/SLURM pensati per Leonardo
- caricano moduli, attivano un venv, controllano GPU e lanciano `python src/main.py ...`
- mostrano il contesto operativo reale del progetto: HPC, non semplice run locale

### `tests/`
- sono utili per capire aspettative e dipendenze del framework
- molti sono smoke test o script diagnostici, non una suite di test unitari molto rigorosa
- la presenza di `test_cluster.py` conferma che Leonardo e il target operativo principale

### `requirements.txt`
- contiene stack LLM (`torch`, `transformers`, `accelerate`, `safetensors`)
- contiene stack notebook/analisi (`pandas`, `matplotlib`, `seaborn`, `ipykernel`)
- contiene utility di sistema e rete

### `requirements_analysis.txt`
- e volutamente minimale per l'analisi risultati

### `.gitignore`
- ignora `src/models`, `venv`, `analysis_venv`, `VAL`
- quindi il progetto previsto includeva modelli locali e validator locali, ma questi componenti non dovevano essere versionati


## 11. Architettura prevista vs snapshot reale

Architettura prevista:
- codice Python completo
- dati PDDL
- prompt specializzati
- modelli locali in `src/models/`
- validator VAL disponibile localmente
- esecuzione su Leonardo
- risultati e analisi notebook

Snapshot reale che hai in workspace:
- codice: presente
- dati: presenti
- prompt: presenti
- risultati: presenti
- notebook: presenti
- `src/models/`: assente
- `VAL/`: assente

Conclusione pratica:
- la snapshot e ottima per studio, reverse engineering e analisi dei risultati gia prodotti
- per rieseguire tutto da zero servono almeno pesi locali dei modelli, validator e ambiente compatibile


In [1]:
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'config.yml').exists() and (candidate / 'src').exists():
            return candidate
    raise FileNotFoundError('Could not locate the project root from the current working directory.')


PROJECT_ROOT = find_project_root()
PROJECT_ROOT


WindowsPath('C:/Users/utente/OneDrive/Documenti/GitHub/LLM_Benchmark/References/LLM-Needs-a-Plan-main/LLM-Needs-a-Plan-main')

In [2]:
import os
from pathlib import Path


def print_tree(root: Path, max_depth: int = 2) -> None:
    root = root.resolve()
    for current, dirs, files in os.walk(root):
        rel = Path(current).relative_to(root)
        depth = 0 if str(rel) == '.' else len(rel.parts)
        if depth > max_depth:
            dirs[:] = []
            continue
        indent = '  ' * depth
        label = root.name if depth == 0 else Path(current).name
        print(f'{indent}{label}/')
        if depth == max_depth:
            continue
        for file_name in sorted(files):
            print(f'{indent}  {file_name}')


print_tree(PROJECT_ROOT, max_depth=2)


LLM-Needs-a-Plan-main/
  .gitignore
  README.md
  Repository Deep Dive.ipynb
  Repository Walkthrough.ipynb
  Results Analysis.ipynb
  config.yml
  report.pdf
  requirements.txt
  requirements_analysis.txt
  assets/
    Guida Leonardo.pdf
    images/
    literature/
    tutorials/
  scripts/
    gemma3_iters_1.sh
    gemma3_iters_2.sh
    gemma3_iters_3.sh
    gemma3_iters_4.sh
    gemma3_iters_4_citycar.sh
    gemma3_iters_4_tetris.sh
    kimi_iters_1.sh
    kimi_iters_2.sh
    kimi_iters_3.sh
    kimi_iters_4.sh
    kimi_iters_4_citycar.sh
    kimi_iters_4_tetris.sh
    llama3_iters_1.sh
    llama3_iters_2.sh
    llama3_iters_3.sh
    llama3_iters_4.sh
    phi4_iters_1.sh
    phi4_iters_2.sh
    phi4_iters_3.sh
    phi4_iters_4.sh
  src/
    main.py
    core/
    data/
    prompts/
    results/
    tests/
    utils/


In [3]:
results_root = PROJECT_ROOT / 'src' / 'results'
rows = []

for plan_file in results_root.rglob('*_plan.txt'):
    content = plan_file.read_text(encoding='utf-8')
    parts = content.split('--- Processing Metadata ---')
    metadata = {}
    if len(parts) > 1:
        for line in parts[1].splitlines():
            if ':' in line:
                key, value = line.split(':', 1)
                metadata[key.strip()] = value.strip()

    rows.append(
        {
            'model': plan_file.parent.parent.name,
            'domain': plan_file.parent.name,
            'valid': metadata.get('Plan Valid', 'False').lower() == 'true',
        }
    )

summary = {}
for row in rows:
    key = (row['model'], row['domain'])
    summary.setdefault(key, {'files': 0, 'valid': 0})
    summary[key]['files'] += 1
    summary[key]['valid'] += int(row['valid'])

for (model, domain), data in sorted(summary.items()):
    rate = (data['valid'] / data['files'] * 100.0) if data['files'] else 0.0
    print(f"{model:7} | {domain:7} | {data['valid']:2}/{data['files']:2} | {rate:5.1f}%")


gemma3  | citycar |  3/20 |  15.0%
gemma3  | tetris  |  4/20 |  20.0%
kimi    | citycar | 16/20 |  80.0%
kimi    | tetris  | 16/20 |  80.0%
llama3  | citycar | 16/20 |  80.0%
llama3  | tetris  | 17/20 |  85.0%
phi4    | citycar |  5/20 |  25.0%
phi4    | tetris  | 13/20 |  65.0%


In [4]:
sample_file = PROJECT_ROOT / 'src' / 'results' / 'gemma3' / 'citycar' / 'instance-02_plan.txt'
content = sample_file.read_text(encoding='utf-8')
plan_text, metadata_text = content.split('--- Processing Metadata ---', 1)

print('Sample file:', sample_file)
print('\nFirst 25 lines of the saved plan block:')
for line in plan_text.splitlines()[:25]:
    print(line)

print('\nMetadata block:')
print(metadata_text.strip())


Sample file: C:\Users\utente\OneDrive\Documenti\GitHub\LLM_Benchmark\References\LLM-Needs-a-Plan-main\LLM-Needs-a-Plan-main\src\results\gemma3\citycar\instance-02_plan.txt

First 25 lines of the saved plan block:
(define (problem citycar-3-3-3)
(diagonal junction0-0 junction1-1)
(diagonal junction1-1 junction0-0)
(diagonal junction0-1 junction1-0)
(diagonal junction1-0 junction0-1)
(diagonal junction0-1 junction1-2)
(diagonal junction1-2 junction0-1)
(diagonal junction0-2 junction1-1)
(diagonal junction1-1 junction0-2)
(diagonal junction1-0 junction2-1)
(diagonal junction2-1 junction1-0)
(diagonal junction1-1 junction2-0)
(diagonal junction2-0 junction1-1)
(diagonal junction1-1 junction2-2)
(diagonal junction2-2 junction1-1)
(diagonal junction1-2 junction2-1)
(diagonal junction2-1 junction1-2)
(starting car0 garage1)
(starting car1 garage0)
(starting car2 garage1)
(arrived car1 junction2-1)
(arrived car2 junction2-0)
(car_start junction0-1 car0 garage1)
(build_straight_oneway junction0